# V_3 · Thời gian và thời tiết

**Mục tiêu:** trả lời câu hỏi nghiên cứu 2: thêm thời tiết cải thiện dự đoán bao nhiêu so với cấu hình A. So sánh A–B dùng cùng mô hình, cùng lưới tham số, cùng mốc chia và cùng 6.086 giờ validation.

**Kịch bản:** dự đoán lưu lượng của một giờ khi đã biết thời gian và thời tiết **quan trắc** của giờ đó. Kết quả không đồng nghĩa với khả năng dự đoán khi chỉ có dự báo thời tiết.

**Chạy trước:** `baseline_V_2.ipynb`, vì notebook này đọc dự đoán của cấu hình A từ `outputs/predictions/V_2_validation.csv`. Test giữ riêng.

## 3.1 · Chuẩn bị dữ liệu

In [1]:
from pathlib import Path
import json
import os
import platform
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import xgboost

# Tìm gốc dự án từ thư mục hiện tại hoặc các thư mục cha.
current_dir = Path.cwd().resolve()
project_dir = next(
    (p for p in (current_dir, *current_dir.parents)
     if (p / "data" / "Metro_Interstate_Traffic_Volume.csv").is_file()),
    None,
)
if project_dir is None:
    raise FileNotFoundError("Không tìm thấy CSV trong thư mục dự án.")
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

output_dir = project_dir / "outputs"
figure_dir = project_dir / "figures"
pd.set_option("display.max_columns", None)
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e5e4e0", "axes.axisbelow": True,
    "lines.linewidth": 2,
})
BLUE, ORANGE = "#2a78d6", "#eb6834"

print("Python:", platform.python_version())
print("pandas:", pd.__version__, "| scikit-learn:", sklearn.__version__, "| xgboost:", xgboost.__version__)

Python: 3.13.9
pandas: 3.0.6 | scikit-learn: 1.9.1 | xgboost: 3.4.1


In [2]:
from src.data import DATA_PATH as data_path
from src.data import load_splits
from src.evaluation import bootstrap_mae_difference, regression_metrics, run_observations
from src.experiment_logger import save_experiment_log
from src.features import CONFIG_B, WEATHER_GROUP_FEATURES, WEATHER_GROUPS, build_features
from src.models import load_predictions, make_model, results_table, run_config, save_results, summarize_search

In [3]:
# Cùng quy tắc với notebooks/preprocessing_features.ipynb và cùng mốc chia với V_1.
splits = load_splits()
features = build_features(splits)
train, validation = features["train"], features["validation"]

assert (len(train), len(validation), len(features["test"])) == (28402, 6086, 6087)
assert train["date_time"].max() < validation["date_time"].min()
display(pd.DataFrame(
    {"so_gio": len(part), "tu_gio": part["date_time"].min(), "den_gio": part["date_time"].max()}
    for part in features.values()
).set_axis(features.keys()))

,so_gio,tu_gio,den_gio
train,28402,2012-10-02 09:00:00,2017-05-10 04:00:00
validation,6086,2017-05-10 05:00:00,2018-01-19 14:00:00
test,6087,2018-01-19 15:00:00,2018-09-30 23:00:00


## 3.2 · Biểu diễn nhãn thời tiết

Sau khi gộp theo giờ, `weather_main` có thể chứa nhiều nhãn như `drizzle | rain`. Phần này kiểm tra lựa chọn giữa one-hot trên chuỗi gộp và cờ 0/1 theo từng nhóm.

In [4]:
combos_train = set(train["weather_main"])
combos_validation = set(validation["weather_main"])
print("Tỷ lệ giờ train có nhiều nhãn:", f"{train['weather_main'].str.contains(' | ', regex=False).mean():.1%}")
print("Số tổ hợp nhãn trong train:", len(combos_train))
print("Số tổ hợp trong validation:", len(combos_validation))
print("Tổ hợp chỉ có trong validation:", sorted(combos_validation - combos_train))

display(pd.DataFrame({
    "train": train[WEATHER_GROUP_FEATURES].sum(),
    "validation": validation[WEATHER_GROUP_FEATURES].sum(),
}).sort_values("train", ascending=False))
assert train[WEATHER_GROUP_FEATURES].sum(axis=1).ge(1).all()

Tỷ lệ giờ train có nhiều nhãn: 12.4%
Số tổ hợp nhãn trong train: 76
Số tổ hợp trong validation: 45
Tổ hợp chỉ có trong validation: ['fog | rain | thunderstorm', 'haze | thunderstorm']


,train,validation
weather_clouds,11129,2120
weather_clear,9135,2115
weather_mist,4011,905
weather_rain,3788,783
weather_snow,1989,362
weather_drizzle,1161,363
weather_haze,895,228
weather_fog,567,162
weather_thunderstorm,507,235
weather_smoke,17,1


**Nhận xét**

- 12,4% số giờ train mang nhiều nhãn `weather_main`. Train có 76 tổ hợp nhãn, validation có 45, trong đó `fog | rain | thunderstorm` và `haze | thunderstorm` chỉ xuất hiện ở validation.
- One-hot trên chuỗi gộp sẽ bỏ qua hai tổ hợp này và tách các tổ hợp hiếm thành cột riêng. Cờ theo 11 nhóm giữ được thông tin của từng nhãn và không gặp nhãn lạ.
- `smoke` (17 giờ train) và `squall` (4 giờ train, 0 giờ validation) quá hiếm để mô hình học ổn định. `snow_1h > 0` chỉ có 31 giờ train và 0 giờ validation; thông tin tuyết chủ yếu nằm ở nhãn `snow`.

## 3.3 · Huấn luyện và tuning cấu hình B

In [5]:
results = run_config("B_thoi_tiet", CONFIG_B, train, validation)
save_results("V_3", results, validation, output_dir)
display(results_table(results).round(3))

,config,model,so_dac_trung,MAE,RMSE,R2,mean_cv_mae
0,B_thoi_tiet,Ridge,27,280.606,450.041,0.948,309.956
1,B_thoi_tiet,Random Forest,27,245.829,399.503,0.959,278.323
2,B_thoi_tiet,XGBoost,27,243.453,389.889,0.961,283.165


## 3.4 · Đối chứng A–B

Chênh lệch dương nghĩa là B có MAE thấp hơn A. Khoảng tin cậy 95% lấy mẫu lại theo ngày trên validation.

In [6]:
previous = load_predictions("V_2", output_dir)
assert previous["date_time"].equals(validation["date_time"])

rows = []
for item in results:
    prediction_a = previous[f"A_lich_le|{item['model']}"].to_numpy()
    metrics_a = regression_metrics(validation["traffic_volume"], prediction_a)
    gap = bootstrap_mae_difference(validation, prediction_a, item["prediction"])
    rows.append({
        "model": item["model"],
        "MAE_A": metrics_a["MAE"], "MAE_B": item["metrics"]["MAE"],
        "RMSE_A": metrics_a["RMSE"], "RMSE_B": item["metrics"]["RMSE"],
        "R2_A": metrics_a["R2"], "R2_B": item["metrics"]["R2"],
        **gap,
    })
ablation = pd.DataFrame(rows)
ablation["giam_MAE_%"] = 100 * ablation["chenh_lech_MAE"] / ablation["MAE_A"]
ablation.to_csv(output_dir / "metrics" / "V_3_ablation_A_B.csv", index=False)
display(ablation.round(3))

,model,MAE_A,MAE_B,RMSE_A,RMSE_B,R2_A,R2_B,chenh_lech_MAE,CI95_thap,CI95_cao,so_ngay,giam_MAE_%
0,Ridge,283.098,280.606,453.231,450.041,0.947,0.948,2.492,-0.497,5.591,255,0.880
1,Random Forest,248.705,245.829,396.713,399.503,0.959,0.959,2.876,-3.352,8.574,255,1.156
2,XGBoost,244.943,243.453,385.920,389.889,0.961,0.961,1.490,-5.135,7.663,255,0.608


Khoảng tin cậy trên chỉ phản ánh dao động theo ngày của validation, chưa phản ánh dao động do `random_state` của XGBoost (lấy mẫu dòng và cột). Cell sau fit lại hai cấu hình với 5 seed, giữ nguyên tham số đã chọn.

In [7]:
from src.features import CONFIG_A
from src.models import load_best_params

params_a = load_best_params("V_2", output_dir)["A_lich_le|XGBoost"]
params_b = load_best_params("V_3", output_dir)["B_thoi_tiet|XGBoost"]
SEEDS = [1, 2, 3, 4, 5]

def seed_maes(columns, params, frame=train):
    """MAE validation của XGBoost với từng seed."""
    values = []
    for seed in SEEDS:
        model = make_model("XGBoost", columns).set_params(**params, random_state=seed)
        model.fit(frame[columns], frame["traffic_volume"])
        values.append(regression_metrics(validation["traffic_volume"], model.predict(validation[columns]))["MAE"])
    return np.array(values)

maes_a = seed_maes(CONFIG_A, params_a)
maes_b = seed_maes(CONFIG_B, params_b)
# Đánh giá thêm A khi dùng đúng tham số của B để loại khác biệt do tuning.
maes_a_same = seed_maes(CONFIG_A, params_b)
display(pd.DataFrame(
    {"MAE_tb": [m.mean() for m in (maes_a, maes_a_same, maes_b)],
     "do_lech_chuan": [m.std(ddof=1) for m in (maes_a, maes_a_same, maes_b)],
     "nho_nhat": [m.min() for m in (maes_a, maes_a_same, maes_b)],
     "lon_nhat": [m.max() for m in (maes_a, maes_a_same, maes_b)]},
    index=["A, tham số của A", "A, tham số của B", "B, tham số của B"],
).round(2))

,MAE_tb,do_lech_chuan,nho_nhat,lon_nhat
"A, tham số của A",244.37,0.39,243.90,244.69
"A, tham số của B",244.37,0.39,243.90,244.69
"B, tham số của B",242.72,0.61,242.22,243.75


**Nhận xét**

- Thêm thời tiết giảm MAE rất ít: XGBoost từ 244,94 xuống 243,45 (0,6%), Random Forest từ 248,71 xuống 245,83 (1,2%), Ridge từ 283,10 xuống 280,61 (0,9%). Khoảng tin cậy của cả ba đều chứa 0.
- RMSE của Random Forest và XGBoost tăng nhẹ (396,7 → 399,5 và 385,9 → 389,9), nên chưa thể nói B tốt hơn A về sai số lớn.
- Với 5 seed, A đạt MAE 244,37 ± 0,39 và B đạt 242,72 ± 0,61. Chênh lệch 1,65 lớn hơn dao động do seed nhưng chỉ khoảng 0,7%. A dùng tham số của A hay của B cho cùng kết quả vì tuning đã chọn cùng một bộ tham số.
- Kết luận trong thiết kế hiện tại: thời tiết quan trắc cùng giờ chỉ bổ sung rất ít khi đã có thời gian và ngày lễ. Kết quả này không có nghĩa thời tiết không ảnh hưởng đến giao thông; mục 3.5 cho thấy tác động tập trung ở một số điều kiện.

## 3.5 · Thời tiết giúp ở điều kiện nào

So sánh XGBoost của A và B theo nhóm thời tiết. Các nhóm chồng lấp vì một giờ có thể mang nhiều nhãn, nên không cộng các dòng trong bảng. Nhóm có ít giờ cần diễn giải thận trọng.

In [8]:
item_b = next(item for item in results if item["model"] == "XGBoost")
prediction_a = previous["A_lich_le|XGBoost"].to_numpy()
rows = []
for name in WEATHER_GROUPS:
    mask = validation[f"weather_{name}"].eq(1).to_numpy()
    if not mask.any():
        continue
    part = validation[mask]
    rows.append({
        "nhom": name, "so_gio": int(mask.sum()),
        "luu_luong_tb": part["traffic_volume"].mean(),
        "MAE_A": np.abs(prediction_a[mask] - part["traffic_volume"]).mean(),
        "MAE_B": np.abs(item_b["prediction"][mask] - part["traffic_volume"]).mean(),
        "sai_so_tb_B": (item_b["prediction"][mask] - part["traffic_volume"]).mean(),
    })
weather_table = pd.DataFrame(rows).sort_values("so_gio", ascending=False)
weather_table["giam_MAE"] = weather_table["MAE_A"] - weather_table["MAE_B"]
display(weather_table.round(1))

print("Số giờ có snow_1h > 0: validation", int(validation["snow_1h"].gt(0).sum()), "| train", int(train["snow_1h"].gt(0).sum()))

# Tập trung vào giờ ban ngày của ngày làm việc, nơi thời tiết xấu có thể làm giảm lưu lượng rõ nhất.
daytime = validation["is_weekend"].eq(0) & validation["is_holiday"].eq(0) & validation["hour"].between(6, 18)
slices = {
    "Mọi giờ ban ngày làm việc": daytime,
    "Có tuyết": daytime & validation["weather_snow"].eq(1),
    "Tuyết lớn (heavy snow)": daytime & validation["weather_description"].str.contains("heavy snow"),
    "Mưa lớn (heavy rain / very heavy rain)": daytime & validation["weather_description"].str.contains("heavy intensity rain|very heavy rain"),
    "Giông": daytime & validation["weather_thunderstorm"].eq(1),
    "Dưới −15 °C": daytime & validation["temp_c"].lt(-15),
}
rows = []
for name, mask in slices.items():
    mask = mask.to_numpy()
    actual = validation.loc[mask, "traffic_volume"].to_numpy()
    rows.append({
        "nhom": name, "so_gio": int(mask.sum()), "so_ngay": validation.loc[mask, "date_time"].dt.normalize().nunique(),
        "MAE_A": np.abs(prediction_a[mask] - actual).mean(),
        "MAE_B": np.abs(item_b["prediction"][mask] - actual).mean(),
        "sai_so_tb_A": (prediction_a[mask] - actual).mean(),
        "sai_so_tb_B": (item_b["prediction"][mask] - actual).mean(),
    })
display(pd.DataFrame(rows).round(1))

part = validation[daytime.to_numpy()]
print("A → B, giờ ban ngày làm việc:", bootstrap_mae_difference(
    part, prediction_a[daytime.to_numpy()], item_b["prediction"][daytime.to_numpy()]))

,nhom,so_gio,luu_luong_tb,MAE_A,MAE_B,sai_so_tb_B,giam_MAE
1,clouds,2120,3954.3,248.3,243.1,-122.3,5.2
0,clear,2115,2914.6,250.3,249.5,-50.7,0.8
5,mist,905,3011.6,221.3,215.9,-69.2,5.4
6,rain,783,3417.3,218.5,220.3,-54.6,-1.8
2,drizzle,363,3375.1,208.8,207.7,-64.4,1.1
8,snow,362,3259.7,336.8,342.6,-44.4,-5.7
9,thunderstorm,235,3137.9,200.4,210.2,-36.6,-9.7
4,haze,228,3297.8,370.6,359.4,-98.1,11.2
3,fog,162,3097.2,198.7,194.7,-129.5,4.0
7,smoke,1,5952.0,82.8,10.6,10.6,72.2


Số giờ có snow_1h > 0: validation 0 | train 31


,nhom,so_gio,so_ngay,MAE_A,MAE_B,sai_so_tb_A,sai_so_tb_B
0,Mọi giờ ban ngày làm việc,2237,173,276.9,262.4,-39.6,-40.5
1,Có tuyết,160,23,491.5,483.6,218.5,34.4
2,Tuyết lớn (heavy snow),1,1,839.6,128.0,839.6,128.0
3,Mưa lớn (heavy rain / very heavy rain),27,17,270.4,255.0,218.7,184.8
4,Giông,75,31,243.9,247.9,28.1,-31.6
5,Dưới −15 °C,111,11,427.0,376.2,343.9,243.1


A → B, giờ ban ngày làm việc: {'chenh_lech_MAE': 14.472559489550733, 'CI95_thap': 5.030545828037897, 'CI95_cao': 25.050195551480094, 'so_ngay': 173}


**Nhận xét**

- Trên toàn validation, mức thay đổi theo nhóm thời tiết nhỏ và không cùng chiều: `haze` giảm 11,2, `clouds` và `mist` giảm khoảng 5, còn `snow` và `thunderstorm` tăng 5,7 và 9,7.
- Khi chỉ xét giờ ban ngày (06:00–18:00) của ngày làm việc không lễ, thời tiết giảm MAE từ 276,9 xuống 262,4. Mức giảm 14,47 xe/giờ có CI 95% từ 5,03 đến 25,05 trên 173 ngày, rõ hơn nhiều so với toàn bộ validation.
- Ở các giờ đó, A dự đoán cao hơn thực tế khi có tuyết (sai số trung bình +218,5 trên 160 giờ / 23 ngày) và khi dưới −15 °C (+343,9 trên 111 giờ / 11 ngày). B giảm các sai lệch này còn +34,4 và +243,1. Thời tiết xấu làm lưu lượng ban ngày giảm và B đã học được một phần.
- Nhóm `heavy snow` chỉ có 1 giờ và mưa lớn có 27 giờ, không đủ để kết luận riêng.

### Giờ ngày lễ và giờ thường

Cell sau tách MAE của A và B theo giờ ngày lễ, đồng thời thử ba mức `colsample_bytree` để kiểm tra chênh lệch có do việc lấy mẫu cột khi số đặc trưng tăng hay không. Mỗi mức chạy 3 seed; các tham số khác giữ theo V_3.

In [9]:
holiday_mask = validation["is_holiday"].eq(1).to_numpy()
actual = validation["traffic_volume"].to_numpy()
rows = []
for colsample in [0.6, 0.8, 1.0]:
    for label, columns in [("A", CONFIG_A), ("B", CONFIG_B)]:
        values = []
        for seed in SEEDS[:3]:
            model = make_model("XGBoost", columns).set_params(
                **{**params_b, "colsample_bytree": colsample}, random_state=seed
            )
            model.fit(train[columns], train["traffic_volume"])
            errors = np.abs(np.clip(model.predict(validation[columns]), 0, None) - actual)
            values.append((errors.mean(), errors[holiday_mask].mean(), errors[~holiday_mask].mean()))
        rows.append({"colsample_bytree": colsample, "cau_hinh": label,
                     **dict(zip(["MAE_tat_ca", "MAE_gio_le", "MAE_gio_thuong"], np.mean(values, axis=0)))})
display(pd.DataFrame(rows).round(1))
print("Số giờ lễ validation:", int(holiday_mask.sum()), "| train:", int(train["is_holiday"].sum()))

,colsample_bytree,cau_hinh,MAE_tat_ca,MAE_gio_le,MAE_gio_thuong
0,0.6,A,244.7,467.6,235.5
1,0.6,B,242.8,575.5,229.2
2,0.8,A,243.6,445.8,235.3
3,0.8,B,242.0,567.8,228.6
4,1.0,A,244.4,433.9,236.6
5,1.0,B,243.9,573.2,230.4


Số giờ lễ validation: 240 | train: 844


**Nhận xét**

- Ở cả ba mức `colsample_bytree`, B có MAE giờ thường thấp hơn A khoảng 6 xe/giờ (228,6–230,4 so với 235,3–236,6), nhưng MAE giờ lễ cao hơn 108–139 xe/giờ (567,8–575,5 so với 433,9–467,6).
- Chênh lệch ở giờ lễ không mất đi khi lấy toàn bộ cột ở mỗi cây, nên nguyên nhân không chỉ là lấy mẫu cột. Train chỉ có 844 giờ lễ; khi có thêm cột thời tiết, mô hình có thể chia nhỏ các giờ lễ theo thời tiết với rất ít mẫu. Đây là giả thuyết, chưa được kiểm chứng.
- Phần lợi ích ở giờ thường bị bù trừ bởi 240 giờ lễ, đây là một lý do khiến tổng MAE của A và B gần nhau. Hướng thử tiếp: tăng `min_child_weight`, hoặc mô hình riêng/ràng buộc cho ngày lễ, đánh giá bằng CV trong train.

## 3.6 · Độ nhạy với các quyết định dữ liệu còn để ngỏ

`data_validation.ipynb` đã ghi nhận các vấn đề chưa có quy tắc xử lý chính thức. Giữ nguyên tham số đã chọn của XGBoost và chỉ thay đổi dữ liệu train; validation không đổi. Đây là phép kiểm tra độ nhạy, chưa phải đề xuất đổi quy tắc.

| Phương án | Nội dung |
|---|---|
| Che nhiệt độ lặp | Coi `temp = 276.793` là thiếu; giá trị này lặp nhiều giờ liên tiếp vào mùa đông 2013–2014 |
| Bổ sung ngày lễ | Thêm các ngày lễ liên bang thiếu nhãn (xem V_2) |
| Bỏ đợt đóng đường | Bỏ các giờ 22/07/2016 20:00 – 24/07/2016 17:00 khỏi train, theo lịch đóng đường trong `docs/station_data_info.docx` |

In [10]:
stuck = load_splits(mask_stuck_temp=True)
changed_train = stuck["train"]["temp"].to_numpy() != splits["train"]["temp"].to_numpy()
print("Số giờ train đổi temp:", int(changed_train.sum()),
      "| validation đổi:", int((stuck["validation"]["temp"].to_numpy() != splits["validation"]["temp"].to_numpy()).sum()))
print("Khoảng thời gian:", splits["train"].loc[changed_train, "date_time"].min(), "đến", splits["train"].loc[changed_train, "date_time"].max())
print("temp trước khi che (K): trung bình", round(splits["train"].loc[changed_train, "temp"].mean(), 2),
      "| sau khi che:", round(stuck["train"].loc[changed_train, "temp"].mean(), 2))

closure = train["date_time"].between("2016-07-22 20:00", "2016-07-24 17:00")
print("Số giờ đóng đường trong train:", int(closure.sum()))

variants = {
    "Quy tắc hiện tại": train,
    "Che nhiệt độ lặp": build_features(stuck)["train"],
    "Bổ sung ngày lễ": build_features(load_splits(complete_holidays=True))["train"],
    "Bỏ đợt đóng đường": train[~closure],
}
rows = []
for name, frame in variants.items():
    maes = seed_maes(CONFIG_B, params_b, frame)
    rows.append({
        "phuong_an": name, "so_gio_train": len(frame),
        "MAE_tb_5_seed": maes.mean(), "do_lech_chuan": maes.std(ddof=1),
        "nho_nhat": maes.min(), "lon_nhat": maes.max(),
    })
sensitivity = pd.DataFrame(rows)
sensitivity["so_voi_hien_tai"] = sensitivity["MAE_tb_5_seed"].iloc[0] - sensitivity["MAE_tb_5_seed"]
display(sensitivity.round(2))

Số giờ train đổi temp: 78 | validation đổi: 0
Khoảng thời gian: 2013-10-22 05:00:00 đến 2014-01-23 08:00:00
temp trước khi che (K): trung bình 276.34 | sau khi che: 261.94
Số giờ đóng đường trong train: 46


,phuong_an,so_gio_train,MAE_tb_5_seed,do_lech_chuan,nho_nhat,lon_nhat,so_voi_hien_tai
0,Quy tắc hiện tại,28402,242.72,0.61,242.22,243.75,0.00
1,Che nhiệt độ lặp,28402,243.12,0.40,242.49,243.51,-0.40
2,Bổ sung ngày lễ,28402,241.56,0.72,240.72,242.56,1.16
3,Bỏ đợt đóng đường,28356,238.61,0.64,238.12,239.73,4.11


**Nhận xét**

- Che giá trị lặp `276.793` đổi 78 giờ train từ 22/10/2013 đến 23/01/2014; nhiệt độ trung bình của các giờ này giảm từ 276,34 K xuống 261,94 K, phù hợp hơn với mùa đông. MAE thay đổi −0,40, nằm trong dao động do seed (độ lệch chuẩn khoảng 0,4–0,7). Quyết định nên dựa vào tính hợp lệ của dữ liệu, không dựa vào MAE.
- Bổ sung ngày lễ giảm 1,16, khoảng 1,6–1,9 lần độ lệch chuẩn theo seed: tín hiệu yếu.
- Bỏ 46 giờ đóng đường giảm MAE trung bình 4,11 (238,61 so với 242,72), lớn hơn nhiều so với dao động do seed. Các giờ này không phản ánh hành vi giao thông thông thường nên có cơ sở để loại khỏi train.
- Cả ba phương án được chọn ra và đo trên cùng validation. Nếu nhóm áp dụng, cần ghi lý do theo dữ liệu, ghi vào quy tắc tiền xử lý và chạy lại toàn bộ phiên bản để giữ so sánh công bằng.

## 3.7 · Ghi nhật ký thực nghiệm

In [11]:
# Log chỉ được tạo khi chủ động bật; đặt biến môi trường SAVE_EXPERIMENT_LOGS=1
# hoặc đổi giá trị bên dưới thành True. Mỗi lần ghi tạo file mới, không ghi đè.
SAVE_EXPERIMENT_LOGS = os.environ.get("SAVE_EXPERIMENT_LOGS") == "1"

common_observations = run_observations(data_path, splits, "notebooks/BASELINE/baseline_V_3.ipynb")
common_problems = ['Dùng thời tiết quan trắc cùng giờ; chưa đo khi chỉ có dự báo thời tiết.', 'Dữ liệu thiếu giờ; validation chỉ gồm các giờ quan sát được.']
next_steps = ['Đánh giá thêm lịch sử lưu lượng trong V_4 và phân tích sai số.']

if SAVE_EXPERIMENT_LOGS:
    for item in results:
        log_path = save_experiment_log(
            model=f"{item['config']} - {item['model']}",
            features=item["features"],
            results=item["metrics"],
            observations=common_observations + [
                "Tập đánh giá: " + item.get("tap_danh_gia", f"validation; {len(validation)} giờ."),
                f"Tham số được chọn: {item['search'].best_params_}",
                f"MAE trung bình CV: {-item['search'].best_score_}",
                f"Kết quả từng fold: {summarize_search(item['search']).to_json(orient='records', force_ascii=False)}",
            ],
            problems=common_problems,
            next_steps=next_steps,
        )
        print("Đã lưu:", log_path.name)
else:
    print("Chưa tạo log. Đặt SAVE_EXPERIMENT_LOGS=1 khi cần ghi.")

Đã lưu: EXP_016.txt
Đã lưu: EXP_017.txt
Đã lưu: EXP_018.txt


### Kết luận V_3

| Mô hình | MAE A | MAE B | Giảm | CI 95% |
|---|---|---|---|---|
| XGBoost | 244,94 | **243,45** | 1,49 | −5,14 đến 7,66 |
| Random Forest | 248,71 | 245,83 | 2,88 | −3,35 đến 8,57 |
| Ridge | 283,10 | 280,61 | 2,49 | −0,50 đến 5,59 |

- Trả lời câu hỏi nghiên cứu 2 trong thiết kế hiện tại: thời tiết cải thiện MAE khoảng 0,6–1,2% trên toàn validation, chưa đủ rõ so với dao động theo ngày. Lợi ích rõ hơn ở giờ ban ngày ngày làm việc (giảm 14,5 xe/giờ), nhưng bị bù trừ ở giờ lễ.
- Demo Time + Weather nên trình bày đúng mức đóng góp này và không mô tả thời tiết là yếu tố chính.
- Còn mở: thời tiết dự báo thay cho quan trắc; quy tắc nhiệt độ lặp và đợt đóng đường cần nhóm chốt.